# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

**Chạy trên Kaggle:** *File → Import Notebook* (tải file này lên) → *Settings*: **Accelerator = GPU T4 x2**,
**Internet = On** → *Run All* (chạy dài thì dùng *Save Version → Save & Run All*).
Code được `git clone` từ GitHub nên **phải `git push` trước** mỗi lần chạy. Mọi kết quả (`runs/`, `predictions/`,
`curves/`, `figures/`, `env.json`) nằm ở `/kaggle/working` → tab *Output*; dữ liệu nằm ở `/kaggle/temp` (không bị lưu).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: Settings → Accelerator = GPU T4 x2, Internet = On.
%pip install -q timm openpyxl

import sys, os, json, platform, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()
ON_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

REPO_URL = "https://github.com/AIVIETNAM-AIO-AnhDinh/K4-Track4-Day2-DinhLenhTienAnh-2A202602928.git"
BRANCH = "main"
SUBMISSION = "submissions/2A202602928_DinhLenhTienAnh"   # thư mục bài nộp trong repo

if ON_KAGGLE or ON_COLAB:
    # Kaggle: /kaggle/working được lưu ở tab Output; /kaggle/temp thì không (dataset, repo để ở đây).
    WORK, SCRATCH = (Path("/kaggle/working"), Path("/kaggle/temp")) if ON_KAGGLE else (Path("/content"), Path("/content"))
    SCRATCH.mkdir(parents=True, exist_ok=True)
    REPO_DIR = SCRATCH / "repo"
    if (REPO_DIR / ".git").exists():
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
    OUT_DIR, DATA_DIR = WORK, SCRATCH / "data"
else:
    # Chạy local: notebook nằm trong <repo>/submissions/<...>/code
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "eval.py").exists())
    OUT_DIR, DATA_DIR = REPO_DIR / SUBMISSION, REPO_DIR / "data"

CODE_DIR = REPO_DIR / SUBMISSION / "code"
assert (REPO_DIR / "eval.py").exists() and (CODE_DIR / "dataset.py").exists(), \
    f"Không thấy eval.py hoặc {CODE_DIR}: đã git push code lên GitHub chưa?"
sys.path.insert(0, str(REPO_DIR))   # để import eval.py
sys.path.insert(0, str(CODE_DIR))   # dataset.py, model.py, train.py ... của bài nộp

RUNS_DIR, PRED_DIR, CURVES_DIR, FIG_DIR = (OUT_DIR / d for d in ("runs", "predictions", "curves", "figures"))
for d in (RUNS_DIR, PRED_DIR, CURVES_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, torch, torchvision, timm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = min(4, os.cpu_count() or 1)
if DEVICE.type == "cuda":
    try:  # P100 (sm_60) không còn được các bản torch mới hỗ trợ: báo sớm thay vì lỗi giữa chừng
        (torch.ones(2, device=DEVICE) * 2).sum().item()
    except RuntimeError as e:
        raise RuntimeError(f"GPU {torch.cuda.get_device_name(0)} không chạy được với torch {torch.__version__}. "
                           "Kaggle: Settings → Accelerator → GPU T4 x2.") from e

git_rev = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                         capture_output=True, text=True).stdout.strip()
ENV = {
    "platform": "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else platform.platform(),
    "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
    "timm": timm.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "cpu_count": os.cpu_count(), "git_commit": git_rev,
}
(OUT_DIR / "env.json").write_text(json.dumps(ENV, indent=2))   # phiên bản thư viện cho README/báo cáo
print(json.dumps(ENV, indent=2))
print("REPO_DIR =", REPO_DIR, "| OUT_DIR =", OUT_DIR, "| DATA_DIR =", DATA_DIR)
if DEVICE.type != "cuda":
    print("CẢNH BÁO: KHÔNG CÓ GPU. Kaggle: Settings → Accelerator → GPU T4 x2.")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
Nếu đã có ảnh (ở local trong `<repo>/images`, hoặc một dataset DeepWeeds gắn vào *Kaggle Input*) thì bỏ qua bước tải.
`IMAGES_DIR` được tìm tự động theo một tên ảnh có thật trong DeepWeeds, không phụ thuộc cấu trúc thư mục trong zip.

In [ ]:
import hashlib, urllib.request, zipfile

ZIP_URL = "https://zenodo.org/records/7939060/files/images.zip?download=1"
ZIP_MD5 = "b7b30f96d466fba86016aa5a26606e0f"
LABELS_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
MARKER = "20160928-140314-0.jpg"   # một ảnh của DeepWeeds, dùng để tìm thư mục ảnh


def find_images_dir(*roots):
    for root in map(Path, roots):
        if root.is_dir():
            hit = next(root.rglob(MARKER), None)
            if hit is not None:
                return hit.parent
    return None


DATA_DIR.mkdir(parents=True, exist_ok=True)
search = [DATA_DIR] + ([Path("/kaggle/input")] if ON_KAGGLE else []) \
         + ([REPO_DIR / "images"] if not (ON_KAGGLE or ON_COLAB) else [])
IMAGES_DIR = find_images_dir(*search)
if IMAGES_DIR is None:
    zip_path = DATA_DIR / "images.zip"
    if not zip_path.exists():
        print("Đang tải images.zip (~490 MB) từ Zenodo ...")
        urllib.request.urlretrieve(ZIP_URL, zip_path)
    h = hashlib.md5()
    with open(zip_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == ZIP_MD5, f"MD5 sai: {h.hexdigest()} (xoá {zip_path} rồi chạy lại)"
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR / "images")
    IMAGES_DIR = find_images_dir(DATA_DIR)

LABELS_DIR = DATA_DIR / "labels"
LABELS_DIR.mkdir(exist_ok=True)
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    dst = LABELS_DIR / f"{name}.csv"
    if not dst.exists():
        urllib.request.urlretrieve(f"{LABELS_URL}/{name}.csv", dst)

n_jpg = sum(1 for _ in IMAGES_DIR.glob("*.jpg"))
print("IMAGES_DIR =", IMAGES_DIR, "|", n_jpg, "ảnh .jpg")
print("LABELS_DIR =", LABELS_DIR, "|", sorted(p.name for p in LABELS_DIR.glob("*.csv")))
assert n_jpg == 17509, f"Kỳ vọng 17.509 ảnh, thấy {n_jpg}"

In [ ]:
# Đường dẫn dùng chung cho mọi thí nghiệm: run(Config(**PATHS, exp_id="B01", backbone="resnet50", seed=0))
PATHS = dict(images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR), out_dir=str(RUNS_DIR),
             pred_dir=str(PRED_DIR), curves_dir=str(CURVES_DIR), num_workers=NUM_WORKERS)
PATHS

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.
Hình và số liệu của bước này được lưu vào `figures/` để đưa vào báo cáo.

### 0.1 Kiểm tra chia dữ liệu và phân bố lớp (README mục 2.1)

In [ ]:
import matplotlib.pyplot as plt
import dataset as D

train_df, val_df, test_df = D.load_split(LABELS_DIR, fold=0)
split_stats = D.check_split(train_df, val_df, test_df, IMAGES_DIR, labels_csv=LABELS_DIR / "labels.csv")

per_class = split_stats["per_class"]
per_class["Δ vs paper"] = per_class["total"] - per_class["paper"]
per_class.to_csv(FIG_DIR / "eda_per_class.csv")
mismatch = split_stats["label_mismatch"]
(FIG_DIR / "split_checks.json").write_text(json.dumps(
    {k: v for k, v in split_stats.items() if k not in ("per_class", "label_mismatch")}
    | {"label_mismatch": mismatch.to_dict("records")}, indent=2, ensure_ascii=False))

c_all, c_tr = per_class["total"], per_class["train"]
print(f"\nMất cân bằng (cả dataset): {c_all.idxmax()} {c_all.max()} / {c_all.idxmin()} {c_all.min()} = "
      f"{c_all.max() / c_all.min():.2f}x | Negatives chiếm {c_all['Negatives'] / c_all.sum():.1%}")
print(f"Mất cân bằng (train): {c_tr.max()} / {c_tr.min()} = {c_tr.max() / c_tr.min():.2f}x")
print("Lệch so với Table 1 của bài báo:", per_class.loc[per_class["Δ vs paper"] != 0, "Δ vs paper"].to_dict())

In [ ]:
share = per_class[["train", "val", "test"]] / per_class[["train", "val", "test"]].sum()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
per_class[["train", "val", "test"]].plot.bar(ax=axes[0], width=0.8)
axes[0].set(title="DeepWeeds fold 0: số ảnh mỗi lớp", ylabel="số ảnh", xlabel="")
for cont in axes[0].containers:
    axes[0].bar_label(cont, fontsize=7, rotation=90, padding=2)
axes[0].set_ylim(0, per_class["train"].max() * 1.15)
(share * 100).plot.bar(ax=axes[1], width=0.8)
axes[1].set(title="Tỉ lệ lớp trong từng tập (%): giống nhau giữa train/val/test", ylabel="% số ảnh của tập", xlabel="")
for ax in axes:
    ax.tick_params(axis="x", rotation=35)
    plt.setp(ax.get_xticklabels(), ha="right")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_class_distribution.png", dpi=150)
plt.show()

### 0.2 Xem ảnh mẫu (3 ảnh mỗi lớp, tập train)

In [ ]:
from PIL import Image

samples = train_df.groupby("Label").sample(n=3, random_state=0).sort_values("Label")
fig, axes = plt.subplots(3, D.NUM_CLASSES, figsize=(2.1 * D.NUM_CLASSES, 7))
for col, (label, grp) in enumerate(samples.groupby("Label")):
    for row, fname in enumerate(grp["Filename"]):
        ax = axes[row, col]
        ax.imshow(Image.open(IMAGES_DIR / fname))
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(fname, fontsize=6)
        if row == 0:
            ax.set_title(f"{label}: {D.CLASS_NAMES[label]}", fontsize=9)
fig.suptitle("DeepWeeds: ảnh mẫu theo lớp (train, fold 0)")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_samples.jpg", dpi=110, pil_kwargs={"quality": 85})
plt.show()

### 0.3 Thống kê ảnh (kích thước, kênh, mean/std) để chọn chuẩn hoá

In [ ]:
from collections import Counter

all_files = pd.concat([train_df, val_df, test_df])["Filename"]
size_mode = Counter()
for f in all_files:
    with Image.open(IMAGES_DIR / f) as im:   # chỉ đọc header, nhanh
        size_mode[(im.size, im.mode)] += 1
print("(kích thước, mode) của toàn bộ ảnh:", dict(size_mode))

# mean/std theo kênh trên 1.000 ảnh train ngẫu nhiên (chỉ train, quy tắc S4)
sub = train_df.sample(n=1000, random_state=0)["Filename"]
px = np.stack([np.asarray(Image.open(IMAGES_DIR / f).convert("RGB"), dtype=np.float32) / 255 for f in sub])
ch_mean, ch_std = px.mean(axis=(0, 1, 2)), px.std(axis=(0, 1, 2))
print("DeepWeeds train mean:", ch_mean.round(3), "std:", ch_std.round(3))
print("ImageNet       mean:", np.round(D.IMAGENET_MEAN, 3), "std:", np.round(D.IMAGENET_STD, 3))
print("→ Dùng mean/std của trọng số tiền huấn luyện (model.data_config), vì backbone học trên phân phối đó.")
del px

### 0.4 Kiểm tra pipeline (slide trang 59, GUIDE.md mục 1.3)
1. cố định seed · 2. loss ban đầu ≈ −ln(1/9) = 2,197 · 3. overfit một batch nhỏ tới loss ≈ 0
· 4. ảnh sau augmentation (giải chuẩn hoá) khớp nhãn · 5. `model.train()` / `model.eval()` đúng lúc.

**Loss ban đầu.** Chạy cho mọi backbone gợi ý (cũng kiểm tra luôn tên model/tag tải được trước Bước 1).
Cột `loss_timm_init` dùng head khởi tạo mặc định của timm, cột `loss` dùng head khởi tạo lại bằng
`model.reset_head` (mặc định của `build_model`, áp dụng giống nhau cho mọi backbone).

In [ ]:
import torch.nn.functional as F
from train import set_seed
from model import (SUGGESTED_BACKBONES, build_model, reset_head, weight_tag, data_config,
                   param_groups, set_train_mode, count_params, count_gmacs)

SEED = 0
LN9 = float(np.log(D.NUM_CLASSES))


def mean_loss(model, loader, n_batches=2):
    model.eval()
    out = []
    with torch.inference_mode():
        for i, (x, y, _) in enumerate(loader):
            if i == n_batches:
                break
            out.append(F.cross_entropy(model(x.to(DEVICE)), y.to(DEVICE)).item())
    return float(np.mean(out))


rows = []
for name in SUGGESTED_BACKBONES.values():
    set_seed(SEED)
    m = build_model(name, pretrained=True, head_std=None).to(DEVICE)   # head mặc định của timm
    cfg_m = data_config(m)
    loader = D.make_loader(val_df, IMAGES_DIR, D.build_transforms(False, 224, mean=cfg_m["mean"], std=cfg_m["std"]),
                           batch_size=64, train=False, num_workers=NUM_WORKERS)
    loss_timm = mean_loss(m, loader)
    reset_head(m)                                                         # như build_model(...) mặc định
    rows.append({"backbone": name, "weight_tag": weight_tag(m), "params_M": round(count_params(m), 2),
                 "GMAC@224": round(count_gmacs(m, 224), 3), "mean": cfg_m["mean"], "std": cfg_m["std"],
                 "loss_timm_init": round(loss_timm, 4), "loss": round(mean_loss(m, loader), 4)})
    del m, loader
    torch.cuda.empty_cache()

init_df = pd.DataFrame(rows)
init_df["|loss - ln9|"] = (init_df["loss"] - LN9).abs().round(4)
init_df.to_csv(FIG_DIR / "check_initial_loss.csv", index=False)
print(f"Kỳ vọng -ln(1/9) = {LN9:.4f} (2 batch val, eval mode)")
display(init_df)
assert (init_df["|loss - ln9|"] < 0.1).all(), "Loss ban đầu lệch xa ln 9: kiểm tra head/nhãn/chuẩn hoá"

**Overfit một batch nhỏ** (2 ảnh mỗi lớp = 18 ảnh, không augmentation) với backbone mốc. Không xuống được gần 0
thì lỗi nằm ở code/model, chưa chạy tiếp.

In [ ]:
CHECK_BACKBONE = "resnet50"   # backbone mốc; đổi để kiểm tra backbone khác
STEPS = 150

set_seed(SEED)
model = build_model(CHECK_BACKBONE, pretrained=True).to(DEVICE)
cfg_m = data_config(model)
MEAN, STD = cfg_m["mean"], cfg_m["std"]
val_tf = D.build_transforms(False, 224, mean=MEAN, std=STD)

tiny = train_df.groupby("Label").sample(n=2, random_state=SEED)
xb, yb, _ = next(iter(D.make_loader(tiny, IMAGES_DIR, val_tf, batch_size=len(tiny), train=False, num_workers=0)))
xb, yb = xb.to(DEVICE), yb.to(DEVICE)

opt = torch.optim.AdamW(param_groups(model, lr_backbone=1e-4, lr_head=1e-3, weight_decay=0.0))
use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
overfit_hist = []
for step in range(STEPS):
    set_train_mode(model)
    with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_amp):
        loss = F.cross_entropy(model(xb), yb)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    overfit_hist.append(loss.item())
    if step % 25 == 0 or step == STEPS - 1:
        print(f"step {step:3d}  loss {loss.item():.4f}")

model.eval()
with torch.inference_mode():
    overfit_acc = (model(xb).argmax(1) == yb).float().mean().item()
print(f"Sau {STEPS} bước: loss train {overfit_hist[-1]:.4f} | accuracy eval-mode trên batch đó {overfit_acc:.0%}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(overfit_hist)
ax.axhline(LN9, ls="--", c="gray", label="ln 9")
ax.set(title=f"Overfit 1 batch ({len(tiny)} ảnh) - {CHECK_BACKBONE}", xlabel="bước", ylabel="loss CE (log)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "check_overfit_one_batch.png", dpi=150)
plt.show()
assert overfit_hist[-1] < 0.05 and overfit_acc == 1.0, "Không overfit được 1 batch: kiểm tra code trước khi chạy tiếp"

**Ảnh sau augmentation.** Hàng trên: batch thật lấy từ train loader (`aug="basic"`, đã giải chuẩn hoá) với nhãn do
loader trả về. Hàng dưới: ảnh gốc đọc lại từ đĩa theo tên file, với nhãn tra trong CSV. Hai hàng phải cùng cây, cùng nhãn.

In [ ]:
train_tf = D.build_transforms(True, 224, aug="basic", mean=MEAN, std=STD)
aug_loader = D.make_loader(train_df, IMAGES_DIR, train_tf, batch_size=8, train=True, num_workers=0, seed=SEED)
xa, ya, fa = next(iter(aug_loader))
csv_label = train_df.set_index("Filename")["Label"]
assert all(csv_label[f] == int(y) for f, y in zip(fa, ya)), "Nhãn của loader không khớp CSV"

fig, axes = plt.subplots(2, len(fa), figsize=(2.3 * len(fa), 5.9))
for j, (img, y, f) in enumerate(zip(D.denormalize(xa, MEAN, STD), ya, fa)):
    axes[0, j].imshow(img.permute(1, 2, 0).numpy())
    axes[0, j].set_title(f"loader: {D.CLASS_NAMES[int(y)]}", fontsize=8)
    axes[1, j].imshow(Image.open(IMAGES_DIR / f))
    axes[1, j].set_title(f"CSV: {D.CLASS_NAMES[csv_label[f]]}", fontsize=8)
    axes[1, j].set_xlabel(f, fontsize=6)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Train loader sau augmentation (trên) và ảnh gốc theo tên file (dưới)")
fig.tight_layout()
fig.savefig(FIG_DIR / "check_augmented_batch.jpg", dpi=110, pil_kwargs={"quality": 85})
plt.show()

# Các mức augmentation của trục B (dùng ở Bước 2) trên cùng một ảnh
img0 = Image.open(IMAGES_DIR / fa[0]).convert("RGB")
modes = [a for a in D.AUG_CHOICES if a != "none"]
fig, axes = plt.subplots(len(modes), 6, figsize=(12, 2.1 * len(modes)))
torch.manual_seed(SEED)
for i, mode in enumerate(modes):
    tf = D.build_transforms(True, 224, aug=mode, mean=MEAN, std=STD)
    for j in range(6):
        axes[i, j].imshow(D.denormalize(tf(img0), MEAN, STD).permute(1, 2, 0).numpy())
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    axes[i, 0].set_ylabel(mode)
fig.suptitle(f"Các mức aug trên {fa[0]} ({D.CLASS_NAMES[int(ya[0])]})")
fig.tight_layout()
fig.savefig(FIG_DIR / "check_aug_modes.jpg", dpi=100, pil_kwargs={"quality": 85})
plt.show()

**`train()` / `eval()`.** Ở eval mode, BatchNorm dùng thống kê đã lưu nên dự đoán của một ảnh không phụ thuộc
các ảnh khác trong batch; ở train mode thì có. Vì vậy mọi lượt đánh giá phải `model.eval()` + `torch.inference_mode()`.

In [ ]:
import copy

probe = copy.deepcopy(model)   # bản sao: forward ở train mode sẽ cập nhật running stats của BN
x8 = xb[:8]
probe.eval()
with torch.inference_mode():
    diff_eval = (probe(x8[:1]) - probe(x8)[:1]).abs().max().item()
probe.train()
with torch.inference_mode():
    diff_train = (probe(x8[:1]) - probe(x8)[:1]).abs().max().item()
print(f"Lệch logit của cùng 1 ảnh khi chạy riêng vs trong batch 8: eval {diff_eval:.2e} | train {diff_train:.2e}")
assert diff_eval < 1e-3 < diff_train
del probe

pipeline_checks = {
    "seed": SEED, "ln9": LN9,
    "initial_loss": init_df.set_index("backbone")[["loss_timm_init", "loss"]].to_dict("index"),
    "overfit": {"backbone": CHECK_BACKBONE, "weight_tag": weight_tag(model), "n_images": len(tiny),
                "steps": STEPS, "final_loss": overfit_hist[-1], "eval_acc": overfit_acc},
    "bn_mode_logit_diff": {"eval": diff_eval, "train": diff_train},
}
(FIG_DIR / "pipeline_checks.json").write_text(json.dumps(pipeline_checks, indent=2))
del model, opt
torch.cuda.empty_cache()
print("Bước 0 xong. Hình và số liệu ở", FIG_DIR)

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# TODO: vòng lặp qua danh sách backbone, mỗi lần gọi train.run(Config(exp_id="B01", backbone=..., seed=0))
#       Gợi ý: from train import Config, run;  run(Config(**PATHS, exp_id="B01", backbone="resnet50", seed=0))
#       Transform phải dùng mean/std của trọng số: data_config(m) trong model.py (ViT augreg dùng 0.5/0.5).
#       Bảng init_df ở Bước 0 đã có tag trọng số, #params, GMAC của các backbone gợi ý.
# TODO: ghi vào bảng: tag trọng số, #params, GMAC, macro-F1 val, top-1 val, thời gian train/epoch.
# TODO: chọn 1-2 backbone đi tiếp, kèm lý do dựa trên số liệu.

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# TODO: các trục A-G của GUIDE.md mục 3 (khởi tạo, augmentation, loss, sampler, LR/optimizer, EMA, ...).
# TODO: thử ít nhất một kết hợp các yếu tố tốt; so Δ với nhiễu (std).

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# TODO: TTA lật, multi-crop/scale, dò độ phân giải, gộp xác suất vs logit, ensemble, EMA/soup,
#       temperature scaling (T khớp trên VAL), gộp BN/FP16  (starter/inference.py)
# TODO: độ trễ p50/p95/p99 bằng starter/benchmark.py: warmup, synchronize, >= 50 lần

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# TODO: chạy chung kết và mốc, mỗi seed, ví dụ:
#   for seed in (0, 1, 2):
#       run(Config(**PATHS, exp_id="F01", ..., seed=seed, save_test_predictions=True))
#       run(Config(**PATHS, exp_id="T00", seed=seed, save_test_predictions=True))
# Với suy luận (TTA/ensemble/temperature scaling), tự ghi predictions/F01_seed<k>_test.csv
# bằng eval.save_predictions(...) từ xác suất cuối cùng; thêm F01uncal_* nếu muốn chấm I4(a).

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "{PRED_DIR}/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out {OUT_DIR}/eval_out
!python {REPO_DIR}/eval.py score --pred "{PRED_DIR}/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out {OUT_DIR}/eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "{PRED_DIR}/F01_seed*_test.csv" \
    --baseline "{PRED_DIR}/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out {OUT_DIR}/eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# TODO: ghi results.xlsx bằng pandas.ExcelWriter(engine="openpyxl") với các sheet:
#   Backbones, Training, Inference, Final, PerClass, Latency, Summary
# TODO: kiểm tra mỗi exp_id có ảnh trong curves/ và số trong xlsx khớp kết quả của eval.py